# 01 — Exploración de la base SEPA

**Objetivo de esta notebook:** entender qué hay adentro antes de calcular nada.

Preguntas que tengo que poder responder al final:
1. ¿Cuántas cadenas, sucursales y productos hay realmente por día?
2. ¿La cobertura es pareja entre provincias o hay sesgo?
3. ¿Los precios tienen la forma que espero (distribución, colas)?
4. ¿Qué proporción de la base cae dentro de mi canasta?


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import duckdb, pandas as pd, matplotlib.pyplot as plt
from src import config as cfg
from src.canasta import cobertura, etiquetar, canasta_df

pd.set_option('display.max_columns', 50)
plt.rcParams.update({'figure.figsize': (11, 5), 'axes.grid': True, 'grid.alpha': .3})
con = duckdb.connect()
PARQUET = str(cfg.INTERIM / 'fecha=*' / '*.parquet')


### 1. Volumen y cobertura por día

In [ ]:
df = con.execute(f"""
    SELECT fecha,
           count(*)                                   AS filas,
           count(DISTINCT ean)                        AS eans,
           count(DISTINCT cadena)                     AS cadenas,
           count(DISTINCT id_comercio || '-' || id_sucursal) AS sucursales,
           count(DISTINCT provincia)                  AS provincias
    FROM read_parquet('{PARQUET}')
    GROUP BY 1 ORDER BY 1
""").df()
display(df)

# Ojo con los días de cobertura baja: si una cadena no reportó, el índice se
# rompe. Ese día hay que excluirlo o imputarlo, y decirlo en el README.
ax = df.plot(x='fecha', y='sucursales', marker='o', legend=False)
ax.set_title('Sucursales que reportaron por día'); ax.set_ylabel('sucursales');

### 2. Concentración: ¿quién domina la base?

Si una sola cadena aporta el 60% de las filas, cualquier promedio simple es en
realidad el precio de esa cadena. Por eso después uso medianas por cadena y no
promedios sobre la base cruda.

In [ ]:
con.execute(f"""
    SELECT cadena, count(*) AS filas,
           round(100.0*count(*) / sum(count(*)) OVER (), 1) AS pct,
           count(DISTINCT id_sucursal) AS sucursales
    FROM read_parquet('{PARQUET}')
    GROUP BY 1 ORDER BY filas DESC
""").df()

### 3. Cobertura geográfica

In [ ]:
prov = con.execute(f"""
    SELECT provincia, count(*) AS filas, count(DISTINCT cadena) AS cadenas
    FROM read_parquet('{PARQUET}') GROUP BY 1 ORDER BY filas DESC
""").df()
display(prov)
# Regla que me impongo: solo comparo precios en provincias con >= 3 cadenas.
print('Provincias comparables:', (prov.cadenas >= 3).sum(), 'de', len(prov))

### 4. Distribución de precios

In [ ]:
px = con.execute(f"SELECT precio FROM read_parquet('{PARQUET}') USING SAMPLE 200000 ROWS").df()
fig, ax = plt.subplots(1, 2)
px.precio.plot.hist(bins=80, ax=ax[0]); ax[0].set_title('Precio (escala lineal)')
px.precio.plot.hist(bins=80, log=True, ax=ax[1]); ax[1].set_title('Precio (log)')
px.precio.describe(percentiles=[.01,.25,.5,.75,.95,.99]).round(1)

### 5. ¿Cuánto de la base cae en mi canasta?

Si la cobertura es muy baja (<5%) el índice es frágil. Si un item captura
demasiados EANs distintos, el patrón está mezclando productos distintos y hay
que corregirlo en `src/canasta.py`.

In [ ]:
muestra = con.execute(f"SELECT ean, descripcion, precio FROM read_parquet('{PARQUET}') USING SAMPLE 300000 ROWS").df()
cob = cobertura(muestra)
display(cob)
print(f"Cobertura de la canasta: {cob.filas.sum()/len(muestra):.1%} de las filas")

### Conclusiones

> Completar con lo que encontraste. Ejemplo:
> - N cadenas, M sucursales, X provincias comparables.
> - La cadena Y concentra Z% de las filas → uso mediana por cadena.
> - La canasta cubre W% de las filas y todos los items tienen >= 5 EANs.
